# MedoraAI Chest X-ray Training - EfficientNet-B4

This notebook trains the `.pt` file used by the MedoraAI backend:

```env
CHEST_MODEL_PATH=./models/chest_xray_efficientnet_b4.pt
```

Backend compatibility target:

```python
timm.create_model("efficientnet_b4", pretrained=True, num_classes=15)
```

The exported file is a PyTorch `state_dict`, so it can be loaded by `backend/services/chest_classifier.py` without changing backend code.

## Important PDF note

The PDFs in `docs/R.P` describe:

- `DATASET.pdf`: CheXpert, 224,316 chest radiographs, 14 observations, uncertainty labels.
- `ALGO.pdf`: Deep AUC Maximization / AUC margin loss, with medical image classification experiments including CheXpert.

The current MedoraAI backend is **not CheXpert-14**. It uses this 15-class NIH ChestX-ray14-style label order:

```text
Atelectasis, Cardiomegaly, Effusion, Infiltration, Mass, Nodule,
Pneumonia, Pneumothorax, Consolidation, Edema, Emphysema, Fibrosis,
Pleural Thickening, Hernia, No Finding
```

So this notebook uses the NIH Chest X-ray14 dataset by default because it matches the backend output head. If you train on CheXpert directly, you must also change `CLASS_LABELS` and `num_classes` in the backend.

In [ ]:
# Runtime setup
!nvidia-smi || true
# Install notebook dependencies. Do not reinstall torch/torchvision here; RunPod images usually include GPU builds.
%pip install -q kaggle timm==0.9.16 pandas numpy pillow tqdm scikit-learn

## Kaggle Credentials and Dataset Storage

This notebook is set up for Jupyter Lab on RunPod or a similar rented GPU.

It first looks for the extracted NIH Chest X-ray14 dataset on the fast local disk:

`/root/nih_chest_xray14_fast`

If that is not ready, it falls back to `/workspace/nih_chest_xray14`. Model outputs still go to `/workspace` so they persist after the pod stops.

Kaggle credentials are only requested if the dataset is missing and the notebook needs to download `data.zip`.


In [ ]:
from pathlib import Path
import os, json, getpass, subprocess

WORKSPACE = Path('/workspace')
FAST_DATA_ROOT = Path('/root/nih_chest_xray14_fast')
WORKSPACE_DATA_ROOT = WORKSPACE / 'nih_chest_xray14'
TMP_ROOT = WORKSPACE / 'tmp'
CACHE_ROOT = WORKSPACE / '.cache'
KAGGLE_DIR = WORKSPACE / '.kaggle'

EXPECTED_PNGS = 112120
MIN_READY_PNGS = 100000

for path in [WORKSPACE_DATA_ROOT, FAST_DATA_ROOT, TMP_ROOT, CACHE_ROOT, KAGGLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ['TMPDIR'] = str(TMP_ROOT)
os.environ['XDG_CACHE_HOME'] = str(CACHE_ROOT)
os.environ['KAGGLE_CONFIG_DIR'] = str(KAGGLE_DIR)


def find_csv(root: Path):
    return list(root.rglob('Data_Entry_2017.csv'))


def count_pngs(root: Path) -> int:
    return sum(1 for _ in root.rglob('*.png'))


def dataset_ready(root: Path) -> bool:
    csvs = find_csv(root)
    pngs = count_pngs(root) if csvs else 0
    return bool(csvs) and pngs >= MIN_READY_PNGS

print('Disk space:')
subprocess.run(['df', '-h', str(WORKSPACE)], check=False)
subprocess.run(['df', '-h', '/'], check=False)

if dataset_ready(FAST_DATA_ROOT):
    DATA_ROOT = FAST_DATA_ROOT
    print('Using extracted dataset on fast local disk:', DATA_ROOT)
elif dataset_ready(WORKSPACE_DATA_ROOT):
    DATA_ROOT = WORKSPACE_DATA_ROOT
    print('Using extracted dataset on workspace volume:', DATA_ROOT)
else:
    DATA_ROOT = FAST_DATA_ROOT
    zip_candidates = [
        FAST_DATA_ROOT / 'data.zip',
        WORKSPACE_DATA_ROOT / 'data.zip',
    ]
    zip_path = next((p for p in zip_candidates if p.exists()), None)

    if zip_path is None:
        kaggle_json = KAGGLE_DIR / 'kaggle.json'
        if not kaggle_json.exists():
            username = input('Paste your Kaggle username: ').strip()
            api_key = getpass.getpass('Paste your Kaggle API key: ').strip()
            if not username or not api_key:
                raise RuntimeError('Kaggle username and API key are required to download the missing dataset')
            kaggle_json.write_text(json.dumps({'username': username, 'key': api_key}))
            os.chmod(kaggle_json, 0o600)

        print('Downloading NIH Chest X-ray14 zip to', WORKSPACE_DATA_ROOT)
        subprocess.run([
            'kaggle', 'datasets', 'download',
            '-d', 'nih-chest-xrays/data',
            '-p', str(WORKSPACE_DATA_ROOT),
        ], check=True)
        zip_path = WORKSPACE_DATA_ROOT / 'data.zip'

    print('Extracting dataset to fast local disk:', DATA_ROOT)
    print('Zip source:', zip_path)
    subprocess.run(['unzip', '-o', str(zip_path), '-d', str(DATA_ROOT)], check=True)

csv_files = find_csv(DATA_ROOT)
png_count = count_pngs(DATA_ROOT)

print('DATA_ROOT:', DATA_ROOT)
print('CSV files:', csv_files[:3])
print('PNG count:', png_count)

if not csv_files:
    raise FileNotFoundError(f'Data_Entry_2017.csv was not found under {DATA_ROOT}')
if png_count < MIN_READY_PNGS:
    raise RuntimeError(f'Only found {png_count} PNG files under {DATA_ROOT}; expected about {EXPECTED_PNGS}')

print('Dataset ready.')


In [ ]:
# Imports and configuration
from pathlib import Path
import random, time, os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

CLASS_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
    'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural Thickening', 'Hernia', 'No Finding'
]

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 3
LR = 1e-4
MAX_IMAGES = 20000  # Set None for full training. Use 1000-5000 for a quick smoke test.

# Keep DATA_ROOT from the dataset setup cell. If this cell is run alone, auto-detect it.
if 'DATA_ROOT' not in globals():
    fast_root = Path('/root/nih_chest_xray14_fast')
    workspace_root = Path('/workspace/nih_chest_xray14')
    DATA_ROOT = fast_root if list(fast_root.rglob('Data_Entry_2017.csv')) else workspace_root

OUT_PATH = Path('/workspace/chest_xray_efficientnet_b4.pt')
print('DATA_ROOT:', DATA_ROOT)
print('OUT_PATH:', OUT_PATH)


In [ ]:
# Build metadata table and image path index
csv_candidates = list(DATA_ROOT.rglob('Data_Entry_2017.csv'))
if not csv_candidates:
    raise FileNotFoundError('Could not find Data_Entry_2017.csv after Kaggle download')
CSV_PATH = csv_candidates[0]
print('CSV:', CSV_PATH)

df = pd.read_csv(CSV_PATH)
print(df.head())
print('Rows:', len(df))

print('Indexing image paths...')
image_paths = {p.name: str(p) for p in DATA_ROOT.rglob('*.png')}
print('Images found:', len(image_paths))

df['path'] = df['Image Index'].map(image_paths)
df = df.dropna(subset=['path']).reset_index(drop=True)
print('Rows with images:', len(df))

if MAX_IMAGES is not None and len(df) > MAX_IMAGES:
    # Patient-level sampling is better for production. This quick subset is for Colab practicality.
    df = df.sample(MAX_IMAGES, random_state=SEED).reset_index(drop=True)
    print('Sampled rows:', len(df))

def encode_labels(label_text):
    labels = set(str(label_text).split('|'))
    return [1.0 if label in labels else 0.0 for label in CLASS_LABELS]

y = np.array([encode_labels(x) for x in df['Finding Labels']], dtype=np.float32)

# Random split for demo training. For publication-grade training, split by Patient ID.
train_idx, val_idx = train_test_split(np.arange(len(df)), test_size=0.15, random_state=SEED)
print('Train:', len(train_idx), 'Val:', len(val_idx))
print(pd.Series(y.sum(axis=0), index=CLASS_LABELS).sort_values(ascending=False))

In [ ]:
class ChestDataset(Dataset):
    def __init__(self, frame, labels, indices, transform):
        self.frame = frame.iloc[indices].reset_index(drop=True)
        self.labels = labels[indices]
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        image = self.transform(image)
        target = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, target

train_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ChestDataset(df, y, train_idx, train_tfms)
val_ds = ChestDataset(df, y, val_idx, val_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# Model exactly matching backend constructor
model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=len(CLASS_LABELS))
model = model.to(DEVICE)

# Weighted BCE helps imbalance. This is the reliable baseline before experimenting with AUC-margin loss.
pos = torch.tensor(y[train_idx].sum(axis=0), dtype=torch.float32)
neg = torch.tensor(len(train_idx) - y[train_idx].sum(axis=0), dtype=torch.float32)
pos_weight = torch.clamp(neg / torch.clamp(pos, min=1.0), max=20.0).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

In [ ]:
def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    all_targets, all_probs = [], []

    for images, targets in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                logits = model(images)
                loss = criterion(logits, targets)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())

    targets = np.concatenate(all_targets)
    probs = np.concatenate(all_probs)
    aucs = []
    for i, label in enumerate(CLASS_LABELS):
        if len(np.unique(targets[:, i])) > 1:
            aucs.append(roc_auc_score(targets[:, i], probs[:, i]))
    return total_loss / len(loader.dataset), float(np.mean(aucs)) if aucs else float('nan')

best_auc = -1
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_auc = run_epoch(train_loader, train=True)
    val_loss, val_auc = run_epoch(val_loader, train=False)
    scheduler.step()

    print(f'Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} train_auc={train_auc:.4f} | val_loss={val_loss:.4f} val_auc={val_auc:.4f} | {time.time()-t0:.1f}s')

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), OUT_PATH)
        print('Saved best:', OUT_PATH, 'AUC:', best_auc)

## Optional: AUC-Maximization Notes From `ALGO.pdf`

`ALGO.pdf` describes Deep AUC Maximization and AUC margin loss. For a robust first local model, the notebook above uses weighted BCE because it is simple and multi-label compatible.

After you have a working baseline, you can experiment with LibAUC. For multi-label chest X-ray, the safest practical pattern is usually one-vs-rest AUC-style training per label or a carefully adapted multi-label loss. Do not switch losses until the BCE pipeline exports a model that the backend can load.

In [ ]:
# Save a small label manifest next to the model for sanity checks
import json
manifest = {
    'backend_expected_model': 'timm efficientnet_b4 num_classes=15',
    'labels': CLASS_LABELS,
    'state_dict_path': str(OUT_PATH),
    'best_val_auc_mean': best_auc,
}
LABELS_PATH = Path('/workspace/chest_xray_efficientnet_b4.labels.json')
LABELS_PATH.write_text(json.dumps(manifest, indent=2))
print(LABELS_PATH.read_text())

In [ ]:
# Copy to MedoraAI model folder on the workspace volume
import shutil
DEST_DIR = Path('/workspace/MedoraAI/models')
DEST_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(OUT_PATH, DEST_DIR / OUT_PATH.name)
shutil.copy2(LABELS_PATH, DEST_DIR / 'chest_xray_efficientnet_b4.labels.json')
print('Saved to:', DEST_DIR)
print('Use locally: CHEST_MODEL_PATH=./models/chest_xray_efficientnet_b4.pt')